# Split Dataset

Randomly splits a folder of images sorted by family into the `training` / `testing` layout that `Train_Model.ipynb` and `Compare_Models_on_Dataset.ipynb` expect.

**What you start with**: a folder inside `Data/` with one subfolder per family:

```
Data/Collection/
├── Carabidae/        *.jpg
├── Chrysomelidae/    *.jpg
└── ...
```

**What you get**: a new dataset folder with every family split into training and testing:

```
Data/<DATASET_NAME>/
├── training/
│   ├── Carabidae/      e.g. 80% of the Carabidae images
│   ├── Chrysomelidae/
│   └── ...
└── testing/
    ├── Carabidae/      the other 20%
    ├── Chrysomelidae/
    └── ...
```

Images are **copied**, so your original folder is left untouched. Each family is split separately, so every family gets the same training/testing ratio.

**How to use it:** edit the settings in Step 1, then choose **Run All**. Nothing past Step 1 needs to change.

## Step 1: Settings

- **`DATASET_NAME`**: the name for the new dataset. It is created as `Data/<DATASET_NAME>/`, and it's the name you'll point `Train_Model.ipynb` at. It must not already exist, so nothing is ever overwritten.
- **`TRAIN_PERCENT`**: the percentage of each family's images that go into `training`. The rest go into `testing`. `80` gives an 80:20 split.
- **`SOURCE_DIR`**: the folder that holds one subfolder per family. Leave it as `None` and a folder picker window pops up when you run the notebook. You can also type the folder's name inside `Data/` (for example `"Collection"` or `"demo_data"`) or its full path.
- **`RANDOM_SEED`**: any whole number. Running again with the same seed and the same images gives exactly the same split. Change it to get a different random split.

In [ ]:
# ======================= EDIT THESE =======================

DATASET_NAME = "My_Dataset"   # created as Data/My_Dataset/

TRAIN_PERCENT = 80            # 80 -> 80% training, 20% testing

SOURCE_DIR = None             # None -> pick with a pop-up, or e.g. "Collection"

RANDOM_SEED = 42              # same seed + same images -> same split

# ==========================================================

## Step 2: Choose the source folder

This finds the repository's `Data/` folder, then opens the folder picker if `SOURCE_DIR` is `None`. The picker sometimes opens behind your editor or browser, so check your taskbar if you don't see it.

To choose a different folder later, re-run the Step 1 cell first.

In [ ]:
import os
import random
import shutil
from collections import Counter

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}


def find_repo_root(marker='Models'):
    path = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.isdir(os.path.join(path, marker)):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            return os.path.abspath(os.getcwd())
        path = parent
    return os.path.abspath(os.getcwd())


def pick_folder(start_dir):
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)  # keep the picker in front of the editor/browser
    folder = filedialog.askdirectory(
        title="Choose the folder that contains one subfolder per family",
        initialdir=start_dir)
    root.destroy()
    return folder


REPO_ROOT = find_repo_root()
DATA_DIR = os.path.join(REPO_ROOT, "Data")

if SOURCE_DIR is None:
    print("A folder picker has opened. Check your taskbar if you don't see it.")
    try:
        SOURCE_DIR = pick_folder(DATA_DIR)
    except Exception as e:
        raise RuntimeError(
            "Couldn't open the folder picker. Instead, type the folder's name into "
            "SOURCE_DIR in Step 1, e.g. SOURCE_DIR = \"Collection\"") from e
    if not SOURCE_DIR:
        raise RuntimeError("No folder was chosen. Run this cell again and pick a folder.")

# A plain name like "Collection" means a folder inside Data/
if not os.path.isabs(SOURCE_DIR):
    SOURCE_DIR = os.path.join(DATA_DIR, SOURCE_DIR)
SOURCE_DIR = os.path.abspath(SOURCE_DIR)

if not os.path.isdir(SOURCE_DIR):
    raise FileNotFoundError(f"This folder doesn't exist: {SOURCE_DIR}")

print("Source folder:", SOURCE_DIR)

## Step 3: Preview the split

This checks your settings, finds every family subfolder and its images (`.jpg`, `.jpeg`, `.png`, `.tif`, `.tiff`, `.bmp`), and shows how many images of each family will go to training and testing. **Nothing is copied yet.**

Every family with at least 2 images keeps at least one image in both training and testing. A family with only 1 image goes entirely into training, and you'll see a warning about it. Empty family folders are skipped.

In [ ]:
def overlaps(a, b):
    """True if folders a and b are the same folder or one is inside the other."""
    a, b = os.path.normcase(a), os.path.normcase(b)
    try:
        return os.path.commonpath([a, b]) in (a, b)
    except ValueError:  # different drives
        return False


# --- Check the settings ---
if isinstance(TRAIN_PERCENT, bool) or not isinstance(TRAIN_PERCENT, (int, float)) or not 0 < TRAIN_PERCENT < 100:
    raise ValueError(f"TRAIN_PERCENT must be a number between 0 and 100 (e.g. 80), not {TRAIN_PERCENT!r}.")

BAD_CHARS = set('<>:"/\\|?*')
DATASET_NAME = str(DATASET_NAME).strip()
if not DATASET_NAME or BAD_CHARS & set(DATASET_NAME):
    raise ValueError(f"DATASET_NAME can't be empty or contain any of these characters: {' '.join(sorted(BAD_CHARS))}")

OUTPUT_DIR = os.path.join(DATA_DIR, DATASET_NAME)
if os.path.exists(OUTPUT_DIR) and os.listdir(OUTPUT_DIR):
    raise FileExistsError(
        f"Data/{DATASET_NAME} already exists and isn't empty. "
        "Choose a different DATASET_NAME in Step 1 so nothing gets overwritten.")
if overlaps(SOURCE_DIR, OUTPUT_DIR):
    raise ValueError(
        f"The source folder and Data/{DATASET_NAME} can't be the same folder or inside one another. "
        "Choose a different source folder or DATASET_NAME.")

# --- Find the families and their images ---
families = {}           # family name -> list of image file names
not_images = Counter()  # family name -> number of other files/folders ignored
for name in sorted(os.listdir(SOURCE_DIR)):
    family_dir = os.path.join(SOURCE_DIR, name)
    if name.startswith(".") or not os.path.isdir(family_dir):
        continue
    images = []
    for f in sorted(os.listdir(family_dir)):
        if os.path.isfile(os.path.join(family_dir, f)) and os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS:
            images.append(f)
        else:
            not_images[name] += 1
    families[name] = images

if not families:
    raise RuntimeError(
        f"No subfolders found in {SOURCE_DIR}. "
        "SOURCE_DIR should be the folder that contains one subfolder per family.")
if {n.lower() for n in families} & {"training", "testing"}:
    raise RuntimeError(
        "This folder contains training/testing folders, so it looks like it has already been split. "
        "Choose the folder that contains one subfolder per family instead.")

empty_families = [name for name, images in families.items() if not images]
families = {name: images for name, images in families.items() if images}
if not families:
    raise RuntimeError(f"None of the family folders in {SOURCE_DIR} contain any images.")

# --- Randomly split each family ---
plan = {}  # family name -> {"training": [...], "testing": [...]}
for family, images in families.items():
    shuffled = list(images)
    # Each family gets its own seeded shuffle, so adding or removing a family
    # doesn't change how the other families are split.
    random.Random(f"{RANDOM_SEED}-{family}").shuffle(shuffled)
    n = len(shuffled)
    n_test = round(n * (100 - TRAIN_PERCENT) / 100)
    if n >= 2:
        n_test = min(max(n_test, 1), n - 1)  # at least one image on each side
    else:
        n_test = 0
    plan[family] = {"training": shuffled[n_test:], "testing": shuffled[:n_test]}

total_train = sum(len(s["training"]) for s in plan.values())
total_test = sum(len(s["testing"]) for s in plan.values())

# --- Show the preview ---
print(f"Source:  {SOURCE_DIR}")
print(f"Output:  {OUTPUT_DIR}")
print(f"Split:   {TRAIN_PERCENT:g}% training / {100 - TRAIN_PERCENT:g}% testing (seed {RANDOM_SEED})\n")

width = max(len("Family"), *(len(f) for f in plan))
line = "-" * (width + 31)
print(f"{'Family':<{width}}  {'Total':>7}  {'Training':>9}  {'Testing':>8}")
print(line)
for family, split in plan.items():
    n_train, n_test = len(split["training"]), len(split["testing"])
    print(f"{family:<{width}}  {n_train + n_test:>7}  {n_train:>9}  {n_test:>8}")
print(line)
print(f"{'All families':<{width}}  {total_train + total_test:>7}  {total_train:>9}  {total_test:>8}")

notes = []
for family in empty_families:
    notes.append(f"WARNING: skipped '{family}' because it has no images.")
for family, split in plan.items():
    if not split["testing"]:
        notes.append(f"WARNING: '{family}' has only 1 image, so it goes into training only.")
for family, count in not_images.items():
    notes.append(f"Note: ignored {count} non-image item(s) inside '{family}'.")
if notes:
    print("\n" + "\n".join(notes))

print("\nIf this looks right, run Step 4 to copy the images.")

## Step 4: Copy the images

This creates `Data/<DATASET_NAME>/training/` and `Data/<DATASET_NAME>/testing/`, copies the images in, and then counts the copied files to confirm everything arrived.

In [ ]:
if os.path.exists(OUTPUT_DIR) and os.listdir(OUTPUT_DIR):
    raise FileExistsError(
        f"Data/{DATASET_NAME} already exists and isn't empty (has this cell already run?). "
        "Choose a different DATASET_NAME in Step 1 and run the notebook again.")

for family, split in plan.items():
    for subset, files in split.items():
        dest_dir = os.path.join(OUTPUT_DIR, subset, family)
        os.makedirs(dest_dir, exist_ok=True)
        for f in files:
            shutil.copy2(os.path.join(SOURCE_DIR, family, f), os.path.join(dest_dir, f))
    print(f"Copied {family}: {len(split['training'])} training, {len(split['testing'])} testing")

# --- Check that every image arrived ---
problems = []
for family, split in plan.items():
    for subset, files in split.items():
        found = len(os.listdir(os.path.join(OUTPUT_DIR, subset, family)))
        if found != len(files):
            problems.append(f"{subset}/{family}: expected {len(files)}, found {found}")
if problems:
    raise RuntimeError("Some images didn't copy correctly:\n" + "\n".join(problems))

print(f"\nDone: {total_train} training and {total_test} testing images copied into {OUTPUT_DIR}")
print("\nTo train on this dataset, set this line in Train_Model.ipynb:")
print(f'    dataset_dir = os.path.join(REPO_ROOT, "Data", "{DATASET_NAME}")')